In [15]:
# imports
import gc
import torch

import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.cuda.amp import autocast, GradScaler
from nltk.translate.bleu_score import corpus_bleu
from rouge_score import rouge_scorer
from tqdm import tqdm


In [3]:
# load the dataframe

df = pd.read_pickle("../../../outputs/stories_with_keywords_train.pkl")
df.head(10)

,text,Keyword 1,Keyword 2,Keyword 3,Keyword 4,Keyword 5
0,"One day, a little girl named Lily found a need...",lily,needle,mom,shirt,share
1,"Once upon a time, there was a little car named...",beep,fuel,leaves,play,day
2,"One day, a little fish named Fin was swimming ...",fin,feel,crab,sun,little
3,"Once upon a time, in a land full of trees, the...",cherry,tree,trees,little,wind
4,"Once upon a time, there was a little girl name...",lily,cobweb,lived,cat,dog
5,"Once upon a time, in a big lake, there was a b...",brown,kayak,tim,water,day
6,"Once upon a time, in a small town, there was a...",lily,triangle,toy,day,saw
7,"Once upon a time, in a peaceful town, there li...",tim,race,sarah,day,park
8,"Once upon a time, there was a clever little do...",max,friends,knee,owl,play
9,"One day, a fast driver named Tim went for a ri...",car,sam,tim,loud,day


In [4]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df['Fold'] = np.tile(np.arange(5), int(np.ceil(len(df) / 5)))[:len(df)]

In [5]:
df["Fold"].value_counts(dropna= False)

Fold
0    423944
1    423944
2    423944
3    423944
4    423943
Name: count, dtype: int64

In [6]:
# Define the Dataset
class KeywordsToStoryDataset(Dataset):
    def __init__(self, dataframe, tokenizer_name="google/flan-t5-base", max_input_length=32, max_target_length=256):
        self.tokenizer = T5Tokenizer.from_pretrained(tokenizer_name)
        self.data = dataframe
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        keywords = ", ".join([str(row[f"Keyword {i}"]) for i in range(1, 6)])
        input_text = f"generate story from keywords: {keywords}"
        target_text = str(row["text"])

        inputs = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        targets = self.tokenizer(
            target_text,
            max_length=self.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": targets["input_ids"].squeeze(),
            "input_text": input_text,
            "target_text": target_text
        }

# Test the Dataset
dataset = KeywordsToStoryDataset(df)

# Print comparisons
print("👉 Original Input Text:", dataset[112]["input_text"])
print("🧠 Decoded Input:", dataset[112]["input_ids"])
print("🔥 Decoded Input:", dataset[112]["attention_mask"])
print("\n🎯 Original Target Text:", dataset[112]["target_text"])
print("🔁 Decoded Target:", dataset[112]["labels"])


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


👉 Original Input Text: generate story from keywords: ben, mama, anna, said, nest
🧠 Decoded Input: tensor([ 3806,   733,    45, 12545,    10,    36,    29,     6, 10786,     6,
            3, 10878,     6,   243,     6,  9190,     1,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0])
🔥 Decoded Input: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

🎯 Original Target Text: Anna and Ben were twins who liked to play outside. They had a big yard with many trees and flowers. One day, they found a bird's nest on a branch. The nest had four blue eggs inside.

"Look, Ben, baby birds!" Anna said, pointing at the nest.

"Can we touch them?" Ben asked, reaching his hand.

"No, Ben, we can't. Mama bird will be sad if we do. She worked hard to make the nest and keep the eggs warm. We have to be kind and leave them alone," Anna said, pulling Ben back.

"Okay, Anna, y

In [13]:
def train_loop_fn(data_loader, model, tokenizer, optimizer, device, scheduler=None):
    model.train()
    running_loss = 0.0

    all_predictions = []
    all_targets = []

    scaler = GradScaler()

    tqdm_ob = tqdm(data_loader, total=len(data_loader), desc="Training")

    for batch_index, batch in enumerate(tqdm_ob):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()

        # Decode predictions and labels
        preds = model.generate(input_ids=input_ids, max_length=256)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        all_predictions.extend(decoded_preds)
        all_targets.extend(decoded_labels)

        del input_ids, attention_mask, labels
        gc.collect()
        torch.cuda.empty_cache()

    # Compute average loss
    train_loss = running_loss / len(data_loader)

    # Compute BLEU score
    references = [[target.split()] for target in all_targets]  # list of list of list of words
    candidates = [pred.split() for pred in all_predictions]    # list of list of words
    bleu_score = corpus_bleu(references, candidates)

    # Compute ROUGE-L score
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [scorer.score(ref, pred)["rougeL"].fmeasure for ref, pred in zip(all_targets, all_predictions)]
    rouge_l_score = sum(rouge_scores) / len(rouge_scores)

    return train_loss, bleu_score, rouge_l_score


In [14]:
def eval_loop_fn(data_loader, model, tokenizer, device):
    model.eval()
    running_loss = 0.0

    all_predictions = []
    all_targets = []

    tqdm_ob = tqdm(data_loader, total=len(data_loader), desc="Evaluating")

    with torch.no_grad():
        for batch_index, batch in enumerate(tqdm_ob):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            running_loss += loss.item()

            preds = model.generate(input_ids=input_ids, max_length=256)
            decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

            all_predictions.extend(decoded_preds)
            all_targets.extend(decoded_labels)

            del input_ids, attention_mask, labels
            gc.collect()
            torch.cuda.empty_cache()

    # Average loss
    val_loss = running_loss / len(data_loader)

    # BLEU
    references = [[target.split()] for target in all_targets]
    candidates = [pred.split() for pred in all_predictions]
    bleu_score = corpus_bleu(references, candidates)

    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [scorer.score(ref, pred)["rougeL"].fmeasure for ref, pred in zip(all_targets, all_predictions)]
    rouge_l_score = sum(rouge_scores) / len(rouge_scores)

    return val_loss, bleu_score, rouge_l_score


In [ ]:
def run():
    # Constants
    EPOCHS = 2
    TRAIN_BATCH_SIZE = 2
    VALID_BATCH_SIZE = 2
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Tokenizer & Model
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
    model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(DEVICE)

    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=2, T_mult=1, eta_min=1e-6, last_epoch=-1
    )

    for fold in range(1, 6):
        print("*" * 20, f"FOLD NUMBER {fold}", "*" * 20)

        df_train = df[df["Fold"] != fold].reset_index(drop=True)
        df_valid = df[df["Fold"] == fold].reset_index(drop=True)

        train_dataset = KeywordsToStoryDataset(df_train)
        valid_dataset = KeywordsToStoryDataset(df_valid)

        train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
        val_loader = DataLoader(valid_dataset, batch_size=VALID_BATCH_SIZE, shuffle=False, num_workers=2)

        for epoch in range(EPOCHS):
            print(f"Epoch --> {epoch + 1} / {EPOCHS}")
            print("-------------------------------")

            train_metrics = train_loop_fn(train_loader, model, tokenizer, optimizer, DEVICE, scheduler)
            print("Training Loss & Metrics:")
            print(f"Loss: {train_metrics[0]:.4f}, BLEU: {train_metrics[1]:.4f}, ROUGE-L: {train_metrics[2]:.4f}")

            val_metrics = eval_loop_fn(val_loader, model, tokenizer, DEVICE)
            print("Validation Loss & Metrics:")
            print(f"Loss: {val_metrics[0]:.4f}, BLEU: {val_metrics[1]:.4f}, ROUGE-L: {val_metrics[2]:.4f}")

        print("\n")

    # Save final model weights
    torch.save(model.state_dict(), '../../../outputs/flan_t5_storygen_5fold_model.pt')


if __name__ == "__main__":
    run()